# Avance 5: Modelo final — Ensambles HAR (Equipo #56)

**Proyecto integrador** — reconocimiento de acciones industriales (InHARD + V-JEPA2).

---

## Objetivos (rúbrica 3.5–3.6)

| Objetivo | Cómo se aborda en esta libreta |
|----------|--------------------------------|
| **3.5** Mejorar rendimiento combinando fortalezas de varios modelos | Ensambles **homogéneos** (voting MLP) y **heterogéneos** (LR+SVM+RF), más **stacking** y **blending** |
| **3.6** Evaluar en datos no vistos | Holdout estratificado 20% sobre embeddings (misma partición para todos los modelos) |
| Optimización de hiperparámetros | `RandomizedSearchCV` (sklearn) + grid sobre MLP PyTorch |
| Tabla comparativa + tiempos | `comparison_table.csv` ordenada por **macro F1** |
| Modelo final + gráficos | ROC, matriz de confusión, PR, importancia de features, calibración/residuos |

**Prerrequisitos:** ejecutar `notebooks/00_Pipeline_Run_All.ipynb` (embeddings + checkpoints fase 4).

**Ejecución:** correr celdas **en orden** de arriba a abajo.


## 1 — Setup

Reproducibilidad, rutas del pipeline y semilla global.


In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 10})

REPO = Path.cwd()
if (REPO / "notebooks").is_dir():
    NB = REPO / "notebooks"
elif REPO.name == "notebooks":
    NB = REPO
else:
    NB = REPO.parent / "notebooks"

sys.path.insert(0, str(NB))
from lib.paths import ensure_notebook_paths, OUTPUTS_DIR, CHECKPOINTS_DIR
from lib.reload import reload_lib_modules

ensure_notebook_paths()
reload_lib_modules()

SEED = 42
QUICK = True  # False → estudio completo (~15–30 min CPU); True → prueba rápida (~5 min)

# embeddings 14 clases (pipeline) o 12 clases filtradas
NPZ = OUTPUTS_DIR / "embeddings.npz"
if not NPZ.is_file():
    alt = OUTPUTS_DIR / "embeddings_train12.npz"
    NPZ = alt if alt.is_file() else NPZ

print("Notebooks:", NB)
print("Embeddings:", NPZ, "→", NPZ.is_file())
print("Checkpoints:", list(CHECKPOINTS_DIR.glob("har_vjepa_*.pt")))


Notebooks: /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/notebooks
Embeddings: /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/notebooks/outputs/embeddings.npz → True
Checkpoints: [PosixPath('/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/notebooks/checkpoints/har_vjepa_train12_100each.pt'), PosixPath('/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/notebooks/checkpoints/har_vjepa_all14_100each.pt'), PosixPath('/Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/notebooks/checkpoints/har_vjepa_all14_5each.pt')]


## 2 — Cargar datos y partición holdout

Los embeddings V-JEPA2 (1024-d) provienen de la fase anterior.  
Usamos **20% holdout estratificado** como *datos no vistos* para comparar modelos individuales y ensambles en igualdad de condiciones.


In [2]:
from lib.har_analysis import load_embeddings_npz
from lib.har_ensemble import holdout_split, ENSEMBLE_DIR

bundle = load_embeddings_npz(NPZ)
X, y = bundle["X"], bundle["y"]
class_names = bundle["class_names"]

X_train, X_test, y_train, y_test = holdout_split(X, y, seed=SEED)
print(f"Muestras: {len(X)} | train={len(y_train)} test={len(y_test)} | clases={len(class_names)}")
print("Clases:", ", ".join(class_names[:4]), "…")


Muestras: 1265 | train=1012 test=253 | clases=14
Clases: Assemble system, Consult sheets, No action, Picking in front …


## 3 — Modelos individuales + optimización de hiperparámetros

Incluye checkpoints de la **fase 4** (`har_vjepa_all14_100each`, `har_vjepa_train12_100each` si aplican) y nuevos clasificadores con búsqueda aleatoria.


In [3]:
from lib.har_ensemble import run_full_ensemble_study

study = run_full_ensemble_study(npz_path=NPZ, quick=QUICK, seed=SEED)
table = study["table"]
table


KeyboardInterrupt: 

## 4 — Tabla comparativa (métrica principal: macro F1)

Modelos ordenados por **macro F1** (robusta al desbalanceo en planta).  
También se reportan accuracy, F1 ponderado, precision/recall macro, AUC macro OVR y **tiempo de entrenamiento**.


In [ ]:
from IPython.display import display

display(table.style.format({
    "accuracy": "{:.1%}",
    "macro_f1": "{:.3f}",
    "weighted_f1": "{:.3f}",
    "macro_precision": "{:.3f}",
    "macro_recall": "{:.3f}",
    "macro_auc_ovr": "{:.3f}",
    "train_sec": "{:.1f}",
}).background_gradient(subset=["macro_f1"], cmap="Greens"))

table.to_csv(study["out_dir"] / "comparison_table.csv", index=False)
print("Guardado →", study["out_dir"] / "comparison_table.csv")


## 5 — Selección del modelo final (criterio de negocio)

**Criterio:** maximizar **macro F1** en holdout (equidad entre meta-acciones raras y frecuentes en piso de producción).  
Restricción opcional: tiempo de reentrenamiento ≤ 120 s para iteraciones ágiles en planta.


In [ ]:
from lib.har_ensemble import select_final_model, PRIMARY_METRIC

final_row = select_final_model(table, max_train_sec=120)
final = study["final"]

print("Métrica principal:", PRIMARY_METRIC)
print("Modelo elegido:", final_row["modelo"], f"({final_row['tipo']})")
print(f"macro F1 = {final_row['macro_f1']:.3f} | accuracy = {final_row['accuracy']:.1%} | train = {final_row['train_sec']:.1f}s")

if final.extra:
    print("Detalle:", final.extra)


## 6 — Gráficos del modelo final + interpretación

| Gráfico | Interpretación |
|---------|----------------|
| Matriz de confusión | Qué meta-acciones se confunden (p. ej. pick vs put) |
| ROC (OvR) | Capacidad de separar cada clase del resto |
| Precision-Recall | Desempeño en clases minoritarias |
| Importancia de features | Dimensiones del embedding V-JEPA más influyentes |
| Calibración / residuos | Si la confianza refleja acierto (útil para alertas en vivo) |


In [ ]:
from IPython.display import Image, display

charts = study["summary"]["charts"]
for title, path in charts.items():
    print("—", title)
    display(Image(filename=path, width=700))


## 7 — Conclusiones

- Se compararon modelos **individuales** (fase 4 + tuning) contra ensambles **homogéneos**, **heterogéneos**, **stacking** y **blending**.
- El holdout estratificado estima desempeño en clips no usados para entrenar en esta libreta.
- **Limitación:** dominio live (YOLO crops) puede diferir; complementar con sesiones en `har_sessions/`.
- **Siguiente paso:** desplegar el ensamble ganador en VisionOps y reforzar clases débiles con HITL.

```python
# Resumen ejecutivo (rellenar tras ejecutar):
# print(study["summary"])
```


In [ ]:
print("Resumen JSON →", study["out_dir"] / "ensemble_summary.json")
study["summary"]
